# KNN — prepoznavanje emocija iz govora

Klasifikacija emocija na RAVDESS skupu podataka pomoću algoritma **K najbližih suseda (KNN)**. Model klasifikuje novi audio zapis na osnovu K najsličnijih zapisa iz trening skupa, gde se sličnost meri rastojanjem u prostoru karakteristika.

## Karakteristike

Svaki audio zapis (3 sekunde, počevši od offset-a 0.5s) se svodi na fiksni vektor od 384 vrednosti: mean, std i max Mel-spektrograma (128 mel-frekvencija × 3 statistike).

## Podela na skupove

Podela je **po glumcu** (actor-independent split): 16 glumaca za trening, 4 za validaciju, 4 za test, bez preklapanja. Isti glumac se nikad ne pojavljuje u više od jednog skupa — model se testira isključivo na glasovima koje nikad nije čuo tokom treninga. Trening skup je dodatno proširen augmentacijom (šum, pomeraj u vremenu, time-stretch, pitch-shift, promena jačine).

## Treniranje

`GridSearchCV` bira hiperparametre (`n_neighbors`, `weights`, `p`) pomoću sopstvene unutrašnje unakrsne provere nad train+val skupom, i automatski trenira finalni model na celom tom skupu. Test skup se koristi isključivo za završnu evaluaciju.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns

X_trainval = np.load("data/processed_data/knn_and_nb/X_trainval.npy")
y_trainval = np.load("data/processed_data/knn_and_nb/y_trainval.npy")
X_test = np.load("data/processed_data/knn_and_nb/X_test.npy")
y_test = np.load("data/processed_data/knn_and_nb/y_test.npy")
le = joblib.load("data/processed_data/knn_and_nb/label_encoder.pkl")

scaler = StandardScaler()
X_trainval_scaled = scaler.fit_transform(np.nan_to_num(X_trainval, nan=0.0))
X_test_scaled = scaler.transform(np.nan_to_num(X_test, nan=0.0))

param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'p': [1, 2],
}
grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(X_trainval_scaled, y_trainval)

best_knn = grid.best_estimator_
print("Best KNN parameters:", grid.best_params_)

y_test_pred = best_knn.predict(X_test_scaled)
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred, target_names=le.classes_))


In [ ]:
os.makedirs("models/knn", exist_ok=True)
joblib.dump(best_knn, "models/knn/knn_model.pkl")
joblib.dump(scaler, "models/knn/scaler.pkl")
joblib.dump(le, "models/knn/label_encoder.pkl")

cm = confusion_matrix(y_test, y_test_pred, labels=np.arange(len(le.classes_)))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('KNN - Test Confusion Matrix')
plt.savefig("models/knn/test_confusion_matrix.png")
plt.show()


## Rezultati

In [ ]:
from IPython.display import Image, display
display(Image(filename="models/knn/test_confusion_matrix.png"))


## Zaključak

KNN model postiže **39.2%** tačnosti na test skupu (actor-independent, glumci koje model nikad nije video). Za poređenje, nasumično pogađanje između 8 emocija daje ~12.5% tačnosti — model je oko 3 puta bolji od slučajnog pogađanja, što pokazuje da statistike Mel-spektrograma nose stvarnu, iskoristivu informaciju o emociji, iako ograničenu.

Najbolje prepoznate emocije su one sa izraženijim akustičkim osobinama (npr. angry, surprise — visoka energija i varijabilnost), dok se emocije sličnog intenziteta i tona (npr. calm/neutral, sad/fear) češće mešaju. Ovo je očekivano ograničenje pristupa koji koristi samo agregirane statistike spektrograma, bez vremenske dinamike.